# MultiGeo-MIA — Kaggle Full Evaluation

**Paper 1: Hidden-State Geometry for Membership Inference**

4-axis unsupervised white-box MIA: magnitude · dimensionality · dynamics · routing

---

### Setup
- **GPU**: H100 80 GB
- **Dataset**: `minh2duy/poisoned-chalice-dataset`
- **Repo**: `technoob05/Poisoned-Chalice-Competition-2026`
- **Models**: 19 WikiMIA + 10 MIMIR + 7 BookMIA

### Workflow
1. Clone repo & install deps
2. Smoke test (tiny model)
3. Full multi-model evaluation on all benchmarks

In [ ]:
# ═══════════════════════════════════════════
# Cell 1: Clone Repo & Install Dependencies
# ═══════════════════════════════════════════
import os, subprocess, sys

REPO_URL = "https://github.com/technoob05/Poisoned-Chalice-Competition-2026.git"
REPO_DIR = "/kaggle/working/repo"

if not os.path.exists(REPO_DIR):
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print(f"✓ Cloned to {REPO_DIR}")
else:
    print(f"✓ Repo already exists at {REPO_DIR}")

# Install dependencies
!pip install -q transformers>=4.40 accelerate datasets scikit-learn scipy huggingface_hub pyarrow

# HuggingFace auth
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    token = UserSecretsClient().get_secret("posioned")
    login(token=token, add_to_git_credential=True)
    print("✓ HuggingFace authenticated")
except Exception as e:
    print(f"○ No HF secret found: {e}")

# Add experiment dir to path
EXP_DIR = os.path.join(REPO_DIR, "Paper1_HiddenStateGeometry", "experiments")
sys.path.insert(0, EXP_DIR)
print(f"✓ sys.path updated: {EXP_DIR}")

In [ ]:
# ═══════════════════════════════════════════
# Cell 2: Verify GPU & Data Paths
# ═══════════════════════════════════════════
import torch

print("=" * 50)
print("  ENVIRONMENT CHECK")
print("=" * 50)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  GPU: {gpu_name} ({vram:.0f} GB)")
else:
    print("  ⚠ No GPU detected!")

KAGGLE_ROOT = "/kaggle/input/datasets/minh2duy/poisoned-chalice-dataset"
for name, sub in [("Poisoned Chalice", "poisoned_chalice_dataset"),
                   ("WikiMIA", "kaggle_wikimia"),
                   ("MIMIR", "kaggle_mimir"),
                   ("BookMIA", "kaggle_bookmia"),
                   ("XSum-MIA", "kaggle_xsum_mia"),
                   ("AGNews-MIA", "kaggle_agnews_mia")]:
    path = os.path.join(KAGGLE_ROOT, sub)
    exists = "✓" if os.path.exists(path) else "✗"
    print(f"  {exists} {name}: {path}")

In [ ]:
# ═══════════════════════════════════════════
# Cell 3: Smoke Test — tiny model
# ═══════════════════════════════════════════
from multigeo import (
    Config, MultiGeoExperiment,
    load_model, free_model,
    MultiGeoExtractor,
    WIKIMIA_MODELS, MIMIR_MODELS, BOOKMIA_MODELS,
)

print(f"✓ Imports OK")
print(f"  WikiMIA models: {len(WIKIMIA_MODELS)}")
print(f"  MIMIR models:   {len(MIMIR_MODELS)}")
print(f"  BookMIA models: {len(BOOKMIA_MODELS)}")

# Load tiny model
print("\nLoading pythia-160m-deduped...")
model, tokenizer, n_layers = load_model("EleutherAI/pythia-160m-deduped", "bfloat16")
cfg_test = Config()
extractor = MultiGeoExtractor(model, tokenizer, n_layers, cfg_test)

features = extractor.extract("def hello():\n    print('Hello, world!')\n")
print(f"\n✓ Extraction OK — {len(features)} features")
for k in ["signal_magnitude", "signal_dimensionality", "signal_dynamics",
          "signal_routing", "loss"]:
    print(f"  {k}: {features[k]:.4f}")

free_model(model, tokenizer, extractor)
print("\n✓ SMOKE TEST PASSED")

In [ ]:
# ═══════════════════════════════════════════
# Cell 4: Poisoned Chalice (Competition)
# ═══════════════════════════════════════════
import gc

cfg = Config()
cfg.output_dir = "/kaggle/working/results_multigeo"
cfg.multi_model = True
cfg.split = "train"  # change to "test" for final submission
os.makedirs(cfg.output_dir, exist_ok=True)

exp = MultiGeoExperiment(cfg)

print("█" * 60)
print("  [1/4] POISONED CHALICE")
print("█" * 60)
pc_results = exp.run_poisoned_chalice()
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ═══════════════════════════════════════════
# Cell 5: WikiMIA (19 models × 4 lengths)
# ═══════════════════════════════════════════
print("█" * 60)
print("  [2/4] WIKIMIA")
print("█" * 60)
wikimia_results = exp.run_wikimia()
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ═══════════════════════════════════════════
# Cell 6: MIMIR (10 models × 7 domains)
# ═══════════════════════════════════════════
print("█" * 60)
print("  [3/4] MIMIR")
print("█" * 60)
mimir_results = exp.run_mimir()
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ═══════════════════════════════════════════
# Cell 7: BookMIA (7 models)
# ═══════════════════════════════════════════
print("█" * 60)
print("  [4/4] BOOKMIA")
print("█" * 60)
bookmia_results = exp.run_bookmia()
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ═══════════════════════════════════════════
# Cell 8: Summary & Export
# ═══════════════════════════════════════════
import pandas as pd

print("═" * 60)
print("  FINAL SUMMARY — MultiGeo-MIA")
print("═" * 60)

summary_rows = []
all_results = {}
if pc_results and "results" in pc_results:
    all_results["PoisonedChalice"] = pc_results
for d in [wikimia_results, mimir_results, bookmia_results]:
    if isinstance(d, dict):
        all_results.update(d)

for key, res in all_results.items():
    if isinstance(res, dict) and "results" in res and len(res["results"]) > 0:
        best = res["results"].iloc[0]
        print(f"  {key:45s}  AUC={best['auc']:.4f}")
        summary_rows.append({
            "benchmark_model": key,
            "best_signal": best.get("score", "N/A"),
            "auroc": float(best["auc"]),
        })

if summary_rows:
    df = pd.DataFrame(summary_rows)
    path = os.path.join(cfg.output_dir, "multimodel_summary.csv")
    df.to_csv(path, index=False)
    print(f"\n  Saved → {path}")
    display(df)

# List output files
print("\n  Output files:")
for f in sorted(os.listdir(cfg.output_dir)):
    sz = os.path.getsize(os.path.join(cfg.output_dir, f)) / 1024
    print(f"    {f} ({sz:.0f} KB)")

print("\n✓ ALL DONE!")